In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['vcards_20200713_165426.vcf', 'GDToT', '20210803_154931.jpg', 'CLUSTER WISE DATA 14TH SACHIVALAYAM.gsheet', 'Contacts.vcf', 'TrustWalletBackup', 'Colab Notebooks', '23BQ1A6123_9_edge detection by using synthectic images (1).gdoc', '23BQ1A6123_9_edge detection by using synthectic images.gdoc', 'python_documentation_2322.gdoc', 'python_documentation_2322.pdf', 'questions-r233cs-Discrete Mathematics and Graph Theory-20240928-1436 (1).gdoc', 'questions-r233cs-Discrete Mathematics and Graph Theory-20240928-1436.gdoc', 'Screenshot_20250716_110058.jpg', 'PSG_AUDIO_PROJECT', 'osa_cnn_transformer_model.h5', 'Week_7_Transfer_Learning_VGG16_block3_и_VGG_19_block5_Colab__2026_st.ipynb', 'Week_7_Transfer_Learning_VGG16_block3_и_VGG_19_block5_Colab__2026_st (1).ipynb', 'ICBHI', 'aru_regulations.pdf', 'RAG.ipynb', 'full_osa_notebook.ipynb', 'osa_cnn_model.h5', 'osa_transformer_model.h5', 'osa_vgg_model.h5', 'Saved from the Google app', 'eurusd_5min_data']


In [ ]:
!pip install mne scipy librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 32.8 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [ ]:
import os
import glob
import numpy as np
import xml.etree.ElementTree as ET
import mne

from scipy.signal import stft, resample_poly

# CONFIG

EDF_DIR = "/content/drive/MyDrive/PSG_AUDIO_PROJECT/raw_temp"
RML_DIR = "/content/drive/MyDrive/PSG_AUDIO_PROJECT/rml_files"

# use a new folder first so old files stay safe
SAVE_DIR = "/content/drive/MyDrive/PSG_AUDIO_PROJECT/processed_mel_v3"

WINDOW_SEC = 5
TARGET_FS = 16000
MIN_OVERLAP = 0.5
N_MELS = 64

# if True, it skips subjects already saved
SKIP_EXISTING = True

os.makedirs(SAVE_DIR, exist_ok=True)

# RML PARSING

def parse_rml(rml_path):
    tree = ET.parse(rml_path)
    root = tree.getroot()

    ns_uri = root.tag.split("}")[0].strip("{")
    ns = {"psg": ns_uri}

    keep = {"Hypopnea", "ObstructiveApnea", "MixedApnea"}
    starts, ends = [], []

    for ev in root.findall(".//psg:Event", ns):
        etype = ev.attrib.get("Type", "")
        if etype in keep:
            s = float(ev.attrib.get("Start", "0"))
            d = float(ev.attrib.get("Duration", "0"))
            starts.append(s)
            ends.append(s + d)

    return np.array(starts, dtype=np.float32), np.array(ends, dtype=np.float32)

# AUDIO LOADING

def load_audio(edf_path, channel="Snore"):
    raw = mne.io.read_raw_edf(
        edf_path,
        include=[channel],
        preload=True,
        verbose="ERROR"
    )
    sig = raw.get_data()[0].astype(np.float32)
    fs = int(raw.info["sfreq"])
    return sig, fs

def downsample(sig, fs, target_fs=16000):
    if fs == target_fs:
        return sig, fs

    from math import gcd
    g = gcd(fs, target_fs)
    up = target_fs // g
    down = fs // g

    sig = resample_poly(sig, up, down).astype(np.float32)
    return sig, target_fs


# WINDOWING

def make_windows(sig, fs, window_sec=5):
    win = int(window_sec * fs)
    n = len(sig) // win
    starts = np.arange(n) * win
    ends = starts + win
    return starts, ends


# LABEL WINDOWS
def label_windows(starts_s, ends_s, ev_s, ev_e, min_overlap=0.5):
    y = np.zeros(len(starts_s), dtype=np.int8)

    for i in range(len(starts_s)):
        ws, we = starts_s[i], ends_s[i]
        max_overlap = 0.0

        for s, e in zip(ev_s, ev_e):
            overlap = max(0.0, min(we, e) - max(ws, s))
            max_overlap = max(max_overlap, overlap)

        if max_overlap >= min_overlap * (we - ws):
            y[i] = 1

    return y

# MEL FILTERBANK

def mel_filterbank(sr, n_fft, n_mels):
    def hz_to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    def mel_to_hz(mel):
        return 700 * (10 ** (mel / 2595) - 1)

    mel_pts = np.linspace(hz_to_mel(0), hz_to_mel(sr / 2), n_mels + 2)
    hz_pts = mel_to_hz(mel_pts)
    bins = np.floor((n_fft + 1) * hz_pts / sr).astype(int)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)

    for i in range(1, n_mels + 1):
        l = bins[i - 1]
        c = bins[i]
        r = bins[i + 1]

        for k in range(l, c):
            fb[i - 1, k] = (k - l) / max(c - l, 1)

        for k in range(c, r):
            fb[i - 1, k] = (r - k) / max(r - c, 1)

    return fb

# LOG-MEL

def compute_logmel(seg, sr, n_mels=64):
    f, t, Z = stft(seg, fs=sr, nperseg=1024, noverlap=768)
    S = np.abs(Z) ** 2
    fb = mel_filterbank(sr, 1024, n_mels)
    mel = fb @ S

    # improved scaling + normalization
    logmel = 10 * np.log10(mel + 1e-10)
    logmel = (logmel - np.mean(logmel)) / (np.std(logmel) + 1e-6)

    return logmel.astype(np.float32)


# FIND ALL SUBJECTS AUTOMATICALLY

def get_all_subjects_from_rml(rml_dir):
    rml_files = sorted(glob.glob(os.path.join(rml_dir, "*.rml")))
    subjects = [os.path.basename(f).replace(".rml", "") for f in rml_files]
    return subjects


# PROCESS ONE SUBJECT

def process_subject(sid, channel="Snore"):
    print("Processing", sid)

    edf_files = sorted(glob.glob(os.path.join(EDF_DIR, f"{sid}*.edf")))
    if len(edf_files) == 0:
        print("No EDF:", sid)
        return False

    rml_path = os.path.join(RML_DIR, sid + ".rml")
    if not os.path.exists(rml_path):
        print("No RML:", sid)
        return False

    sig_all = []
    final_fs = None

    for edf in edf_files:
        sig, fs = load_audio(edf, channel=channel)
        sig, fs = downsample(sig, fs, TARGET_FS)
        sig_all.append(sig)
        final_fs = fs

    sig = np.concatenate(sig_all)

    starts, ends = make_windows(sig, final_fs, WINDOW_SEC)
    win_start_s = starts / final_fs
    win_end_s = ends / final_fs

    ev_s, ev_e = parse_rml(rml_path)
    y = label_windows(win_start_s, win_end_s, ev_s, ev_e, MIN_OVERLAP)

    X = []
    for s, e in zip(starts, ends):
        seg = sig[s:e]
        X.append(compute_logmel(seg, final_fs, N_MELS))

    X = np.stack(X).astype(np.float32)

    np.save(os.path.join(SAVE_DIR, sid + "_Xmel.npy"), X)
    np.save(os.path.join(SAVE_DIR, sid + "_y.npy"), y)

    print("Saved:", sid, "| X shape:", X.shape, "| Apnea windows:", int(y.sum()), "| Total windows:", len(y))
    return True

# RUN ALL SUBJECTS

subjects = get_all_subjects_from_rml(RML_DIR)

print("Total subjects found from RML:", len(subjects))
print("First 10 subjects:", subjects[:10])

done = 0
skipped = 0
failed = 0

for sid in subjects:
    x_out = os.path.join(SAVE_DIR, sid + "_Xmel.npy")
    y_out = os.path.join(SAVE_DIR, sid + "_y.npy")

    if SKIP_EXISTING and os.path.exists(x_out) and os.path.exists(y_out):
        print("Skipping already processed:", sid)
        skipped += 1
        continue

    try:
        ok = process_subject(sid)
        if ok:
            done += 1
        else:
            failed += 1
    except Exception as e:
        print("Failed:", sid, e)
        failed += 1

print("\nFinished")
print("Processed:", done)
print("Skipped  :", skipped)
print("Failed   :", failed)
print("Saved in :", SAVE_DIR)

Total subjects found from RML: 80
First 10 subjects: ['00000995-100507', '00000999-100507', '00001000-100507', '00001006-100507', '00001008-100507', '00001010-100507', '00001014-100507', '00001018-100507', '00001020-100507', '00001022-100507']
Skipping already processed: 00000995-100507
Skipping already processed: 00000999-100507
Skipping already processed: 00001000-100507
Skipping already processed: 00001006-100507
Skipping already processed: 00001008-100507
Processing 00001010-100507
No EDF: 00001010-100507
Processing 00001014-100507
No EDF: 00001014-100507
Processing 00001018-100507
No EDF: 00001018-100507
Processing 00001020-100507
No EDF: 00001020-100507
Processing 00001022-100507
No EDF: 00001022-100507
Skipping already processed: 00001024-100507
Processing 00001026-100507
No EDF: 00001026-100507
Skipping already processed: 00001028-100507
Skipping already processed: 00001037-100507
Skipping already processed: 00001039-100507
Skipping already processed: 00001041-100507
Skipping a

In [ ]:
import os
print(len(os.listdir(SAVE_DIR)))


142


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import glob

files = glob.glob("/content/dataset/**/*.wav", recursive=True)

plt.figure(figsize=(15, 10))

for i, file in enumerate(files[:16]):  # 16 images grid
    y, sr = librosa.load(file, sr=16000)

    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    S_db = librosa.power_to_db(S, ref=np.max)

    plt.subplot(4, 4, i+1)
    librosa.display.specshow(S_db, sr=sr)
    plt.title(file.split("/")[-1][:10])
    plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1500x1000 with 0 Axes>